# MURI: Modelling actuator self-assembly

## Import and start Polyscope

First, import statements

In [7]:
import numpy as np
import triangle as tr
import polyscope as ps
import fabsim_py
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

Then, generate the 2D mesh for tissue

In [8]:
vertices = [
[4.116003, 1.403990],
[4.034014, 1.537785],
[4.176056, 1.259017],
[3.678986, 1.841007],
[3.534014, 1.901057],
[3.932104, 1.657107],
[3.812784, 1.759017],
[3.381433, 1.937688],
[3.224997, 1.950000],
[0.000000, 1.950000],
[4.224998, 0.000000],
[4.224998, 0.950000],
[4.212687, 1.106434],
[-4.116003, 1.403990],
[-4.034014, 1.537785],
[-4.176056, 1.259017],
[-3.678986, 1.841007],
[-3.534014, 1.901057],
[-3.932104, 1.657107],
[-3.812784, 1.759017],
[-3.381433, 1.937688],
[-3.224997, 1.950000],
[-4.224998, 0.000000],
[-4.224998, 0.950000],
[-4.212687, 1.106434],
[4.116003, -1.403990],
[4.034014, -1.537785],
[4.176056, -1.259017],
[3.678986, -1.841007],
[3.534014, -1.901057],
[3.932104, -1.657107],
[3.812784, -1.759017],
[3.381433, -1.937688],
[3.224997, -1.950000],
[0.000000, -1.950000],
[4.224998, -0.950000],
[4.212687, -1.106434],
[-4.116003, -1.403990],
[-4.034014, -1.537785],
[-4.176056, -1.259017],
[-3.678986, -1.841007],
[-3.534014, -1.901057],
[-3.932104, -1.657107],
[-3.812784, -1.759017],
[-3.381433, -1.937688],
[-3.224997, -1.950000],
[-4.224998, -0.950000],
[-4.212687, -1.106434]
]
segments = [[11,12],[5,6],[7,8],[3,4],[3,6],[8,9],[0,1],[0,2],[4,7],[1,5],[10,11],[2,12],[23,24],[18,19],[20,21],[16,17],[16,19],[21,9],[13,14],[13,15],[17,20],[14,18],[22,23],[15,24],[35,36],[30,31],[32,33],[28,29],[28,31],[33,34],[25,26],[25,27],[29,32],[26,30],[10,35],[27,36],[46,47],[42,43],[44,45],[40,41],[40,43],[45,34],[37,38],[37,39],[41,44],[38,42],[22,46],[39,47]]
n_boundary = len(vertices)
n_circle = 16
def make_circle(radius, centerX, centerY):
  n = len(vertices)
  for i in range(n_circle):
    vertices.append([centerX + radius * np.cos(2 * i / n_circle * np.pi), centerY + radius * np.sin(2 * i / n_circle * np.pi)])
    segments.append([n + i, n + ((i + 1) % n_circle)])

make_circle(0.75, -4.55/2, 0)
make_circle(0.7, 4.55/2, 0)
holes = [[-2.25, 0], [2.25, 0]]

A = dict(vertices=vertices, segments=segments, holes=holes)
B = tr.triangulate(A, 'pqa0.1')

zeros = np.zeros((len(B['vertices']), 1))
P = np.hstack((np.array(B['vertices']), zeros))
F = np.array(B['triangles'])

Show initial mesh

In [ ]:
# Initialize polyscope
ps.init()
ps.set_give_focus_on_show(True)
ps.set_ground_plane_mode("shadow_only")

ps_mesh = ps.register_surface_mesh("Initial mesh", P, F, smooth_shade=True, enabled=True)
ps.set_navigation_style("planar")
ps.set_view_projection_mode("orthographic")
ps.set_SSAA_factor(3)
ps.set_view_from_json('{"farClipRatio":20.0,"fov":16.0,"nearClipRatio":0.005,"projectionMode":"Orthographic","viewMat":[1.0,-0.0,0.0,-0.0,0.0,0.997785151004791,-0.0665190145373344,0.000246047973632812,-0.0,0.0665190145373344,0.997785151004791,-25.5602951049805,0.0,0.0,0.0,1.0],"windowHeight":1964,"windowWidth":3456}')
ps.show()

[polyscope] Backend: openGL3_glfw -- Loaded openGL version: 4.1 Metal - 88.1


## Display compaction with polar plots

In [11]:
def polygons(polymer_frac, scale):
  n = polymer_frac.shape[1]
  polygon = []
  rotated_polygon = []
  for i in range(n):
    angle = np.pi / n * i
    polygon.append(np.array([np.cos(angle), np.sin(angle), 0.0]))
    rotated_polygon.append(np.array([np.cos(angle + np.pi), np.sin(angle + np.pi), 0.0]))
  polygon = scale * np.array(polygon)
  rotated_polygon = scale * np.array(rotated_polygon)

  # faces = np.arange(F.shape[0] * 2 * n).reshape(F.shape[0], 2 * n)
  polygon_face = np.zeros((2 * n, 3))
  polygon_face[:, 1] = np.arange(1, 2 * n + 1)
  polygon_face[:, 2] = np.arange(2, 2 * n + 2)
  polygon_face[2 * n - 1, 2] = 1

  verts = []
  faces = []
  for i in range(F.shape[0]):
    center = (V[F[i, 0], :] + V[F[i, 1], :] + V[F[i, 2], :]) / 3
    verts.append(center.reshape((1, 3)))
    verts.append(polygon * polymer_frac[i, :][:, None] + center)
    verts.append(rotated_polygon * polymer_frac[i, :][:, None] + center)
    faces.append(polygon_face + i * (2 * n + 1))
  verts = np.concatenate(verts, axis=0)
  verts[:, 2] = 0.1
  faces = np.concatenate(faces, axis=0)

  return verts, faces


In [ ]:
V = P.copy()

fixed_dofs = []
for i in range(n_boundary, n_boundary + len(holes) * n_circle):
  fixed_dofs.append(3 * i)
  fixed_dofs.append(3 * i + 1)
  fixed_dofs.append(3 * i + 2)

Constants

In [13]:
dt = 1 / 24
k0 = 1e-4
k1 = 5e-2
kd = 0.1 # rate of dissociation
e0 = 1.2e-1
e1 = 1.7e-1
frac_f = 0.7
frac_s = 0.25
n = 16

Iterative compaction in 20 steps

In [ ]:
polymer_frac = np.zeros((F.shape[0], n))
for k in range(20):
  stretch_factor = 1 + 0.5 * (k + 1) / 10.
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
  ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255))
  sigmas = 1.0 * fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  
  phi = fabsim_py.polymer_fraction_reduced(sigmas, k1 / k0, kd / k0, frac_f, frac_s)

  for i in range(1000):
    fabsim_py.polymer_fraction_one_step(polymer_frac, sigmas, k0, k1, kd, frac_f, frac_s, dt)
    fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
    sigmas = 1.0 * fabsim_py.fiber_stress(V, P / stretch_factor, F, n)

  verts, faces = polygons(polymer_frac, 0.5)
  ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
  ps.show()
  # ps.screenshot()

KeyboardInterrupt: 

## Steady-state solution for fiber distribution

In [ ]:
stretch_factor = 1.9

polymer_frac = np.zeros((F.shape[0], n))
fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)


Polar plot of result in face #392

## Convergence of solutions

Compare convergence of ODE and steady-state solution

## Nonlinear least-squares fitting for ODE coefficients

In [17]:
def fitting(strains, phi_measured):
  def fun(params):
    k1 = params[0]
    kd = params[1]
    e0 = params[2]
    e1 = params[3]

    stress = 1.0 * fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
    phi = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

    return (phi - phi_measured).flatten()

  initial_guess = np.array([5e-2, 0.1, 1.2e-1, 1.7e-1])
  return least_squares(fun, initial_guess)

In [ ]:
phi_measured = polymer_frac + 0.01 * np.random.default_rng().random(polymer_frac.shape)
params = fitting(strains, phi_measured)

Polar plots of initial, with noise, and recovered